# Scoped T1 MVP — results and demonstration

Run `mvp-t1-003`. This notebook **reads published evidence and runs inference only**. It never
trains, never rewrites evidence, and never touches sealed TEST.

**Scope.** A one-layer session-context GRU on the frozen C1 example slice, trained for task
T1 (next distinct category) only, over a 1,000-client pilot population, and scored by the
frozen evaluator on all 438,185 validation decisions. This is a preliminary demonstrator.
It is not the original multi-task MVP commitment and not a Phase-1 freeze.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd()
if not (ROOT / "ppsi").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

RUN = "mvp-t1-003"
PUBLIC = ROOT / "docs/evidence/mvp" / RUN
PRIVATE = ROOT / "artifacts/mvp" / RUN


def evidence(name):
    return json.loads((PUBLIC / name).read_text(encoding="utf-8"))


report = evidence("report_inputs.v1.json")
comparison = evidence("comparison.v1.json")
verification = evidence("verification.v1.json")
print("run:", RUN)
print("verification:", verification["status"], f"({len(verification['checks'])} checks)")

## 1. The matched result table

Every row below is scored by the same frozen evaluator over the same validation decisions
with the same support, so the rows are directly comparable.

In [ ]:
rows = report["result_table"]
width = max(len(r["system"]) for r in rows)
print(f"{'system'.ljust(width)}  {'family':<15} {'raw macro MRR@20':>17} {'clients':>9}")
for row in rows:
    print(
        f"{row['system'].ljust(width)}  {row['family']:<15} "
        f"{row['t1_next_distinct_mrr_at_20_macro']:>17.5f} {row['support_clients']:>9,}"
    )

delta = report["headline_comparison"]["fedavg_minus_centralized"]
print(f"\nFedAvg minus centralized: {delta:+.5f}")
print("computed between:", report["headline_comparison"]["computed_between"])

## 2. Fixed-round curve, counts and measured payload

The three client counts are different quantities. The pool is 1,000; the number of
round-client pairs is 1,000; the number of distinct people actually trained is smaller; and
the SuperNode slot count is a simulation detail, not a study population.

In [ ]:
curve = report["validation_curve"]
print(f"{'round':>6} {'centralized':>13} {'fedavg':>10}")
for left, right in zip(curve["centralized"], curve["fedavg"]):
    print(f"{left['server_round']:>6} {left['value']:>13.5f} {right['value']:>10.5f}")

population = report["population"]
print()
for key in (
    "declared_pilot_population",
    "scheduled_participations",
    "measured_unique_clients_trained",
    "supernode_execution_slots",
):
    print(f"{key:>34}: {population[key]:,}")

communication = report["communication"]
print()
for key in ("download_bytes", "upload_bytes", "total_bytes"):
    print(f"{key:>34}: {communication[key] / 1e9:,.2f} GB")
print(f"{'boundary':>34}: {report['communication_boundary']}")

## 3. Inference demonstration on two synthetic sessions

The two sessions below are **invented for illustration**. They are not real users, not drawn
from the dataset, and not evaluation evidence. They exist to show that the trained weights
load and produce a ranked category distribution through the same batch builder the training
used.

In [ ]:
from ppsi.data.batching import windows_to_batch
from ppsi.data.sequences import Windows
from ppsi.federated.mvp_runner import build_trainer, model_identity
from ppsi.models.batch_spec import CATEGORY_PAD, EVENT_PAD, PRICE_BAND_PAD

policy = json.loads((ROOT / "config/mvp/execution.v3.json").read_text(encoding="utf-8"))
identity = model_identity(policy)
HISTORY = int(policy["data"]["history_length"])

# Two predeclared synthetic sessions, written as dense category codes only.
SYNTHETIC = {
    "browses one category, then a neighbour": [41, 41, 41, 57],
    "wanders across four categories": [12, 200, 87, 301, 15],
}


def synthetic_windows(sessions):
    count = len(sessions)
    category = np.full((count, HISTORY), CATEGORY_PAD, dtype="int32")
    event = np.full((count, HISTORY), EVENT_PAD, dtype="int8")
    band = np.full((count, HISTORY), PRICE_BAND_PAD, dtype="int8")
    product = np.zeros((count, HISTORY), dtype="int32")
    brand = np.zeros((count, HISTORY), dtype="int32")
    gap = np.zeros((count, HISTORY), dtype="float32")
    lengths = np.zeros(count, dtype="int64")
    for row, codes in enumerate(sessions.values()):
        length = len(codes)
        lengths[row] = length
        category[row, :length] = codes
        event[row, :length] = 1  # 'view' in the model-lane encoding
        band[row, :length] = 1
        product[row, :length] = 1
        brand[row, :length] = 1
        gap[row, 1:length] = 0.5
    last = np.array([codes[-1] for codes in sessions.values()], dtype="int32")
    return Windows(
        category=category,
        product=product,
        event=event,
        brand=brand,
        price_band=band,
        gap=gap,
        lengths=lengths,
        target=np.zeros(count, dtype="int64"),
        query_category=last,
        query_product=np.ones(count, dtype="int32"),
        query_brand=np.ones(count, dtype="int32"),
        query_price_band=np.ones(count, dtype="int8"),
        client=np.arange(count, dtype="int64"),
    )


demo_batch = windows_to_batch(
    synthetic_windows(SYNTHETIC), np.arange(len(SYNTHETIC)), identity.spec, validate=True
)
print("batch built and validated:", tuple(demo_batch.t1_target.shape))

In [ ]:
def load_regime(name):
    model, _ = build_trainer(identity, policy, None)
    if name == "centralized":
        payload = torch.load(
            PRIVATE / "centralized/checkpoints/round_20.pt",
            map_location="cpu",
            weights_only=False,
        )
        state = payload["model_state_dict"]
    else:
        state = torch.load(
            PRIVATE / "federated/aggregate_round_20.pt",
            map_location="cpu",
            weights_only=True,
        )
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


models = {name: load_regime(name) for name in ("centralized", "fedavg")}

with torch.no_grad():
    logits = {name: model(demo_batch).t1_logits for name, model in models.items()}

for row, label in enumerate(SYNTHETIC):
    print(f"\nsynthetic session: {label}")
    print(f"  history (dense category codes): {SYNTHETIC[label]}")
    for name, values in logits.items():
        top = torch.topk(values[row], 5)
        pairs = ", ".join(
            f"{int(code)} ({float(score):.2f})" for code, score in zip(top.indices, top.values)
        )
        print(f"  {name:>12} top-5: {pairs}")

### 3.1 What the two checkpoints agree on

A small agreement summary between the selected round-20 states. This is a demonstration of
inference behaviour, not a deployment claim and not an evaluation result: the frozen metrics
in section 1 are the only quality evidence.

In [ ]:
for row, label in enumerate(SYNTHETIC):
    central_top = torch.topk(logits["centralized"][row], 5).indices.tolist()
    fed_top = torch.topk(logits["fedavg"][row], 5).indices.tolist()
    shared = sorted(set(central_top) & set(fed_top))
    print(f"{label}: top-5 overlap = {len(shared)}/5 {shared}")

flat_central = logits["centralized"].flatten()
flat_fed = logits["fedavg"].flatten()
correlation = float(torch.corrcoef(torch.stack([flat_central, flat_fed]))[0, 1])
print(f"\nlogit correlation across both synthetic sessions: {correlation:.4f}")

## 4. What this notebook deliberately does not show

- **T2 and T3 demonstrations were not run.** The joint T1/T2 checkpoint lives in an external
  archive that was not available in this environment, so no hash or role verification was
  possible and nothing was loaded. The T3 retrieval demonstration was likewise not executed;
  the existing measured T3 evidence stands as context only.
- No real user identifier, session key or raw event appears anywhere above.
- Synthetic sessions are illustrations. They carry no evidential weight.

## 5. Limitations

- Session-local context only: predictions see at most the current session's last 20 events.
  There is no persistent cross-session personalization.
- Validation-only pilot. TEST was never opened, and no TEST number is claimed.
- Federated clients are logical data partitions in a simulation, not physical devices.
- Reported bytes are serialized model payload at the Flower message boundary, not complete
  wire traffic.
- No differential privacy, no secure aggregation, no device isolation.
- Absolute quality sits below the count baselines. That is a property of a 20-round pilot on
  roughly 32,000 training rows, and it is reported rather than tuned away.